# Baseline: Emotion-LLaMA-v2 tren test split cua Vie-GameEmo

**Muc dich:** chay thu (zero-shot, khong finetune) model **Emotion-LLaMA-v2** tren tap
`data/labels/test.json` cua project `vie-gameemo-skeleton`, de co mot baseline so sanh
voi model dual-path tu xay dung.

**Input:**
- Video: `vie-gameemo-skeleton/data/raw_videos/*.mp4` (can duoc upload len may vast.ai truoc,
  thu muc nay hien dang rong trong repo)
- Label: `vie-gameemo-skeleton/data/labels/test.json` (518 clip, 8 nhan: neutral/amused/hype/
  tilted/sad/shocked/fear/disgusted)

**Yeu cau moi truong (vast.ai hoac may Linux GPU bat ky):**
- GPU >= 24GB VRAM (Llama-2-7b fp16 ~14GB + EVA-ViT-g + Whisper-large-v3 + overhead)
- ~30GB dung luong dia trong cho checkpoints
- Tai khoan HuggingFace da **accept license** cua `meta-llama/Llama-2-7b-chat-hf`
  (https://huggingface.co/meta-llama/Llama-2-7b-chat-hf) + mot **HF token** (read scope)
- ffmpeg co san (dung cho trich audio tu video)

**Luu y quan trong ve nhan (label mismatch):**
Emotion-LLaMA-v2 duoc train voi 7 nhan Ekman: `anger, disgust, fear, happy, neutral, sad,
surprise` (prompt "[emotion]" goc trong `app.py`). Con test set cua vie-gameemo dung 8 nhan
gaming-specific: `neutral, amused, hype, tilted, sad, shocked, fear, disgusted`. Hai khong gian
nhan **khac nhau**, nen notebook nay anh xa (map) nhan goc -> nhan gan nghia nhat trong khong
gian 7-class cua model de tinh accuracy/F1 (xem `GOLD_TO_MODEL_LABEL` o Cell 2 — day la mot gia
dinh, ban co the chinh lai neu thay khong hop ly, vi du `hype` co the map sang `surprise` thay
vi `happy` tuy vao cach hieu).

In [ ]:
# ============================================================
# CELL 1 -- Duong dan & moi truong
# ============================================================
import os, sys, subprocess
from pathlib import Path
import torch

NOTEBOOK_DIR = Path.cwd()

# Gia dinh cau truc mac dinh: hai repo la sibling trong cung mot thu muc goc
# (giong local: c:/Projects/emo_project/{Emotion-LLaMA-v2, vie-gameemo-skeleton})
# Chinh lai 2 bien duoi day neu layout tren may vast.ai khac.
BASE_DIR = Path(os.environ.get("EMO_PROJECT_DIR", NOTEBOOK_DIR.parent.parent))
EMOTION_LLAMA_DIR = Path(os.environ.get("EMOTION_LLAMA_DIR", BASE_DIR / "Emotion-LLaMA-v2"))
VIE_GAMEEMO_DIR = Path(os.environ.get("VIE_GAMEEMO_DIR", BASE_DIR / "vie-gameemo-skeleton"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", BASE_DIR / "models"))
MODELS_DIR.mkdir(parents=True, exist_ok=True)

REPO_URLS = {
    EMOTION_LLAMA_DIR: "https://github.com/ooochen-30/Emotion-LLaMA-v2.git",
    VIE_GAMEEMO_DIR: "https://github.com/rhy221/vie_gameemo.git",
}
for repo_dir, url in REPO_URLS.items():
    if not repo_dir.exists():
        print(f"Khong thay {repo_dir}, dang git clone tu {url} ...")
        subprocess.run(["git", "clone", "--depth=1", url, str(repo_dir)], check=True)

assert (EMOTION_LLAMA_DIR / "minigpt4").exists(), f"Sai duong dan EMOTION_LLAMA_DIR: {EMOTION_LLAMA_DIR}"
assert (VIE_GAMEEMO_DIR / "data" / "labels" / "test.json").exists(), \
    f"Khong thay data/labels/test.json trong {VIE_GAMEEMO_DIR}"

print(f"EMOTION_LLAMA_DIR = {EMOTION_LLAMA_DIR}")
print(f"VIE_GAMEEMO_DIR   = {VIE_GAMEEMO_DIR}")
print(f"MODELS_DIR        = {MODELS_DIR}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | {gpu.total_memory / 1e9:.1f} GB VRAM")
else:
    print("CANH BAO: khong tim thay GPU -- model 7B se rat cham/khong chay duoc tren CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ============================================================
# CELL 2 -- Cau hinh chay
# ============================================================
RAW_VIDEOS_DIR = VIE_GAMEEMO_DIR / "data" / "raw_videos"
LABELS_PATH = VIE_GAMEEMO_DIR / "data" / "labels" / "test.json"
OUTPUT_DIR = VIE_GAMEEMO_DIR / "outputs" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "emotion_llama_v2_test_predictions.jsonl"
METRICS_PATH = OUTPUT_DIR / "emotion_llama_v2_test_metrics.json"
CONFUSION_MATRIX_PATH = OUTPUT_DIR / "emotion_llama_v2_test_confusion_matrix.png"

# Dat so nguyen (vd. 20) de smoke-test truoc khi chay full 518 clip (co the mat vai gio tren 1 GPU)
LIMIT = None

MAX_NEW_TOKENS = 30

# Prompt goc dung cho task phan loai cam xuc, lay tu quick_prompts["Emotion"] trong app.py
TASK = "[emotion]"
INSTRUCTION = "Identify the emotion expressed in the video. Choose from: anger, disgust, fear, happy, neutral, sad, surprise."

MODEL_LABELS = ["anger", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

# Anh xa 8 nhan cua vie-gameemo -> 7 nhan Ekman ma Emotion-LLaMA-v2 co the sinh ra (zero-shot).
# Day la mot gia dinh -- chinh lai neu can (vd. "hype" -> "surprise" thay vi "happy").
GOLD_TO_MODEL_LABEL = {
    "neutral": "neutral",
    "amused": "happy",
    "hype": "happy",
    "tilted": "anger",
    "sad": "sad",
    "shocked": "surprise",
    "fear": "fear",
    "disgusted": "disgust",
}

# Checkpoints se tai tu HuggingFace
LLAMA_REPO_ID = "meta-llama/Llama-2-7b-chat-hf"
LLAMA_DIR = MODELS_DIR / "Llama-2-7b-chat-hf"

EMOTION_LLAMA_CKPT_REPO_ID = "ooochen/Emotion-LLaMA-v2"
EMOTION_LLAMA_CKPT_FILENAME = "stage2.pth"  # checkpoint da finetune (stage 2); doi sang "stage1.pth" neu muon dung ban pretrain
CKPT_PATH = MODELS_DIR / "emotion_llama_v2" / EMOTION_LLAMA_CKPT_FILENAME

# QUAN TRONG: duong dan nay bi hardcode trong my_utils/extract_features.py cua Emotion-LLaMA-v2
# (khong doi duoc qua config) -- phai tai whisper-large-v3 vao dung day.
WHISPER_HARDCODED_PATH = Path("/home/user/big_space/models/openai/whisper-large-v3")

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # dan token vao day hoac set env var HF_TOKEN truoc khi mo notebook

print(f"LIMIT (smoke test) = {LIMIT}")
print(f"Predictions se luu tai: {PREDICTIONS_PATH}")

In [ ]:
# ============================================================
# CELL 3 -- Cai dat thu vien
# ============================================================
import subprocess

subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

get_ipython().system(f'pip install -q -r "{EMOTION_LLAMA_DIR / "requirement.txt"}"')
get_ipython().system('pip install -q huggingface_hub scikit-learn pandas matplotlib seaborn tqdm')

print("Done")

In [ ]:
# ============================================================
# CELL 4 -- Dang nhap HuggingFace & tai checkpoints
# ============================================================
from huggingface_hub import login, snapshot_download, hf_hub_download

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN rong -- can token HuggingFace (read scope) da accept license "
        "cua meta-llama/Llama-2-7b-chat-hf. Dat bien HF_TOKEN o CELL 2 hoac export "
        "env var HF_TOKEN truoc khi mo notebook."
    )
login(token=HF_TOKEN)

# 1) Llama-2-7b-chat-hf (gated -- can accept license tren huggingface.co truoc)
if not LLAMA_DIR.exists() or not any(LLAMA_DIR.iterdir()):
    print(f"Dang tai {LLAMA_REPO_ID} -> {LLAMA_DIR} ...")
    snapshot_download(
        repo_id=LLAMA_REPO_ID,
        local_dir=str(LLAMA_DIR),
        ignore_patterns=["*.bin", "*.bin.index.json"],  # uu tien safetensors, bo .bin trung lap
    )
else:
    print(f"Da co: {LLAMA_DIR}")

# 2) Checkpoint Emotion-LLaMA-v2 (stage2, da finetune)
if not CKPT_PATH.exists():
    print(f"Dang tai {EMOTION_LLAMA_CKPT_REPO_ID}/{EMOTION_LLAMA_CKPT_FILENAME} ...")
    CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    downloaded = hf_hub_download(
        repo_id=EMOTION_LLAMA_CKPT_REPO_ID,
        filename=EMOTION_LLAMA_CKPT_FILENAME,
        local_dir=str(CKPT_PATH.parent),
    )
    print(f"-> {downloaded}")
else:
    print(f"Da co: {CKPT_PATH}")

# 3) Whisper-large-v3 -- BAT BUOC nam dung duong dan hardcode trong my_utils/extract_features.py
if not WHISPER_HARDCODED_PATH.exists() or not any(WHISPER_HARDCODED_PATH.iterdir()):
    print(f"Dang tai openai/whisper-large-v3 -> {WHISPER_HARDCODED_PATH} ...")
    WHISPER_HARDCODED_PATH.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id="openai/whisper-large-v3", local_dir=str(WHISPER_HARDCODED_PATH))
else:
    print(f"Da co: {WHISPER_HARDCODED_PATH}")

# 4) EVA-ViT-g: my_utils/extract_features.py va minigpt4/models/eva_vit.py deu tu dong tai
#    (qua torch hub cache) tu Google Storage khi model duoc khoi tao lan dau -- khong can lam gi them o day.

print("\nHoan tat tai checkpoints.")

## Buoc 1 -- Ghi eval config & load model

In [ ]:
# ============================================================
# CELL 5 -- Ghi eval config (yaml) & load model Emotion-LLaMA-v2
# ============================================================
import yaml

sys.path.insert(0, str(EMOTION_LLAMA_DIR))

eval_cfg = {
    "model": {
        "arch": "emotion_llama_v2",
        "model_type": "pretrain",
        "max_txt_len": 2000,
        "end_sym": "</s>",
        "low_resource": False,
        "prompt_template": "[INST] {} [/INST]",
        "llama_model": str(LLAMA_DIR),
        "ckpt": str(CKPT_PATH),
        "lora_r": 64,
        "lora_alpha": 16,
    },
    "datasets": {
        "mer2023": {
            "vis_processor": {"train": {"name": "blip2_image_eval", "image_size": 448}},
            "text_processor": {"train": {"name": "blip_caption"}},
        }
    },
}
EVAL_CFG_PATH = EMOTION_LLAMA_DIR / "eval_configs" / "vie_gameemo_test.yaml"
with open(EVAL_CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(eval_cfg, f, sort_keys=False, allow_unicode=True)
print(f"Da ghi eval config: {EVAL_CFG_PATH}")

from argparse import Namespace
from minigpt4.common.eval_utils import init_model, prepare_texts
from minigpt4.conversation.conversation import CONV_VISION_minigptv2
from my_utils.extract_features import FeatureExtractor, extract_frame

args = Namespace(cfg_path=str(EVAL_CFG_PATH), options=[])
model, vis_processor = init_model(args)
model.eval()

feature_extractor = FeatureExtractor(device=device)

print("Model da san sang.")

## Buoc 2 -- Load nhan test & kiem tra video

In [ ]:
# ============================================================
# CELL 6 -- Load data/labels/test.json & kiem tra video ton tai
# ============================================================
import json

with open(LABELS_PATH, encoding="utf-8") as f:
    test_labels = json.load(f)

print(f"Tong so sample trong test.json: {len(test_labels)}")

samples = []
missing = []
for item in test_labels:
    video_path = RAW_VIDEOS_DIR / item["video"]
    if video_path.exists():
        samples.append({**item, "video_path": video_path})
    else:
        missing.append(item["video"])

print(f"Video tim thay: {len(samples)} / {len(test_labels)}")
if missing:
    print(f"CANH BAO: thieu {len(missing)} video, vi du: {missing[:5]}")
    print(f"-> Kiem tra da upload du video vao {RAW_VIDEOS_DIR} chua.")

if LIMIT is not None:
    samples = samples[:LIMIT]
    print(f"LIMIT={LIMIT} -- chi chay tren {len(samples)} sample dau tien (smoke test)")

## Buoc 3 -- Chay inference tren toan bo test set (resumable)

Vong lap duoi day ghi ket qua ra file JSONL ngay sau moi video (khong doi den cuoi), nen neu
notebook bi ngat giua chung, chay lai CELL 7 se **tu dong bo qua** cac video da xu ly roi.

In [ ]:
# ============================================================
# CELL 7 -- Vong lap inference (resumable)
# ============================================================
import time
import gc
from tqdm.auto import tqdm

def parse_predicted_label(response_text):
    if not response_text:
        return None
    text = response_text.lower()
    for label in MODEL_LABELS:
        if label in text:
            return label
    return None

def run_single_inference(video_path):
    image = extract_frame(str(video_path))
    if image is None:
        return None

    image_tensor = vis_processor(image).unsqueeze(0).to(feature_extractor.device)
    video_feature = feature_extractor.extract_eva_vit_g_features(str(video_path))
    if video_feature is None:
        return None
    audio_feature = feature_extractor.extract_whisper_audio_features(str(video_path))
    if audio_feature is None:
        # Video khong co audio track -- van chay duoc, coi audio feature la 0
        audio_feature = torch.zeros(1, 64, 1280)

    video_feature = video_feature.to(feature_extractor.device)
    audio_feature = audio_feature.to(feature_extractor.device)

    conv = CONV_VISION_minigptv2.copy()
    conv.system = ""
    texts = prepare_texts([f"{TASK} {INSTRUCTION}"], conv)

    with torch.no_grad():
        response = model.generate(
            image_tensor, video_feature, audio_feature, texts,
            max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        )[0]
    return response

# Nap lai cac video da xu ly (resume)
done_videos = set()
if PREDICTIONS_PATH.exists():
    with open(PREDICTIONS_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                done_videos.add(json.loads(line)["video"])
    print(f"Da xu ly truoc do: {len(done_videos)} video -- se bo qua")

todo = [s for s in samples if s["video"] not in done_videos]
print(f"Con lai: {len(todo)} video")

errors = []
with open(PREDICTIONS_PATH, "a", encoding="utf-8") as out_f:
    for i, sample in enumerate(tqdm(todo, desc="Emotion-LLaMA-v2 inference")):
        try:
            t0 = time.perf_counter()
            response = run_single_inference(sample["video_path"])
            elapsed = time.perf_counter() - t0
            predicted_label = parse_predicted_label(response)

            record = {
                "id": sample["id"],
                "video": sample["video"],
                "gold_label": sample["choice"],
                "raw_response": response,
                "predicted_label": predicted_label,
                "elapsed_sec": round(elapsed, 2),
            }
            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_f.flush()
        except Exception as e:
            errors.append({"video": sample["video"], "error": str(e)})
            print(f"LOI xu ly {sample['video']}: {e}")

        if (i + 1) % 20 == 0:
            gc.collect()
            torch.cuda.empty_cache()

print(f"\nHoan tat. Loi: {len(errors)}")
if errors:
    print("Danh sach video loi (5 dau tien):", errors[:5])

## Buoc 4 -- Metrics & phan tich ket qua

In [ ]:
# ============================================================
# CELL 8 -- Tinh accuracy / macro-F1 / classification report
# ============================================================
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_json(PREDICTIONS_PATH, lines=True)
print(f"Tong so prediction: {len(df)}")

df["gold_mapped"] = df["gold_label"].map(GOLD_TO_MODEL_LABEL)

n_no_pred = df["predicted_label"].isna().sum()
if n_no_pred:
    print(f"CANH BAO: {n_no_pred} response khong parse duoc nhan hop le (raw_response khong "
          f"chua tu nao trong {MODEL_LABELS})")

eval_df = df.dropna(subset=["predicted_label", "gold_mapped"])
print(f"So sample dung de tinh metric: {len(eval_df)} / {len(df)}")

y_true = eval_df["gold_mapped"]
y_pred = eval_df["predicted_label"]

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", labels=MODEL_LABELS, zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", labels=MODEL_LABELS, zero_division=0)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")
print()
print(classification_report(y_true, y_pred, labels=MODEL_LABELS, zero_division=0))

In [ ]:
# ============================================================
# CELL 9 -- Confusion matrix
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

cm = confusion_matrix(y_true, y_pred, labels=MODEL_LABELS)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues",
            xticklabels=MODEL_LABELS, yticklabels=MODEL_LABELS, ax=ax, cbar_kws={"label": "Ty le hang"})
ax.set_xlabel("Predicted (Emotion-LLaMA-v2)")
ax.set_ylabel("Gold (vie-gameemo, da map)")
ax.set_title(f"Confusion Matrix -- Accuracy={accuracy:.3f}, Macro F1={macro_f1:.3f}")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150)
plt.show()
print(f"Da luu -> {CONFUSION_MATRIX_PATH}")

In [ ]:
# ============================================================
# CELL 10 -- Luu tom tat metrics
# ============================================================
summary = {
    "model": "Emotion-LLaMA-v2 (stage2, zero-shot)",
    "n_total_labels": len(test_labels),
    "n_videos_found": len(samples),
    "n_predictions": len(df),
    "n_unparsed_predictions": int(n_no_pred),
    "n_evaluated": len(eval_df),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "model_labels": MODEL_LABELS,
    "gold_to_model_label_map": GOLD_TO_MODEL_LABEL,
    "prompt": f"{TASK} {INSTRUCTION}",
    "n_errors": len(errors),
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Da luu metrics -> {METRICS_PATH}")
print(json.dumps(summary, ensure_ascii=False, indent=2))